<a href="https://colab.research.google.com/github/DivyaMeenaSundaram/Prompt-Engineering/blob/main/LangSmith_Prompt_Management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-google-genai langsmith python-dotenv

In [3]:
# ============================================================
# STEP 2: LOAD API KEYS FROM COLAB SECRETS
# ============================================================
# We use Colab's Secret Manager instead of writing API keys
# directly in the notebook. This prevents the actual keys
# from appearing in the code that we may later share.
# ============================================================

from google.colab import userdata

# Retrieve the Gemini API key stored in Colab Secrets.
gemini_api_key = userdata.get("Gemini_key")

# Retrieve the LangSmith API key stored in Colab Secrets.
langsmith_api_key = userdata.get("Langsmith_key")

# Confirm that the keys were retrieved WITHOUT displaying
# the actual secret values.
print("Gemini API key loaded:", bool(gemini_api_key))
print("LangSmith API key loaded:", bool(langsmith_api_key))

Gemini API key loaded: True
LangSmith API key loaded: True


In [4]:
# ============================================================
# STEP 3: CONFIGURE LANGSMITH TRACING
# ============================================================
# These environment variables tell LangChain to send
# execution traces to our LangSmith account.
#
# LANGSMITH_TRACING:
#     Turns tracing ON.
#
# LANGSMITH_API_KEY:
#     Authenticates our application with LangSmith.
#
# LANGSMITH_PROJECT:
#     Gives our traces a project name so that we can easily
#     find them later in the LangSmith dashboard.
# ============================================================

import os

# Enable LangSmith tracing.
os.environ["LANGSMITH_TRACING"] = "true"

# Provide the LangSmith API key retrieved from Colab Secrets.
os.environ["LANGSMITH_API_KEY"] = langsmith_api_key

# Give this experiment/application its own LangSmith project.
os.environ["LANGSMITH_PROJECT"] = "Prompt-Management-Demo"

# Tell the Gemini integration which API key to use.
os.environ["GOOGLE_API_KEY"] = gemini_api_key

print("LangSmith tracing:", os.environ["LANGSMITH_TRACING"])
print("LangSmith project:", os.environ["LANGSMITH_PROJECT"])
print("Gemini API key configured:", bool(os.environ["GOOGLE_API_KEY"]))

LangSmith tracing: true
LangSmith project: Prompt-Management-Demo
Gemini API key configured: True


In [5]:
# ============================================================
# STEP 4: CHECK THE LANGSMITH CONNECTION
# ============================================================
# This cell checks whether the LangSmith Python SDK can
# authenticate using the API key stored in Colab Secrets.
# We do not display the API key itself.
# ============================================================

from langsmith import Client

# Create a LangSmith client using the configured API key.
langsmith_client = Client(api_key=langsmith_api_key)

print("LangSmith client created successfully.")
print("Target project:", os.environ["LANGSMITH_PROJECT"])

LangSmith client created successfully.
Target project: Prompt-Management-Demo


In [6]:
# ============================================================
# STEP 5.1: IMPORT THE REQUIRED LANGCHAIN COMPONENTS
# ============================================================
# We import:
#
# ChatPromptTemplate:
#     Used to create a structured prompt with variables.
#
# ChatGoogleGenerativeAI:
#     Connects LangChain to Google's Gemini models.
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

print("LangChain components imported successfully.")

LangChain components imported successfully.


In [7]:
# ============================================================
# STEP 5.2: CREATE THE FIRST PROMPT
# ============================================================
# This is a deliberately simple customer-service prompt.
#
# The {question} placeholder will be replaced with the
# customer's actual question when the chain is executed.
#
# Later, we will create prompt versions such as V1, V2,
# and V3 and compare their executions in LangSmith.
# ============================================================

prompt = ChatPromptTemplate.from_template(
    """
    You are a customer service assistant.

    Answer the customer's question clearly and concisely.

    Customer question:
    {question}
    """
)

print("Prompt created successfully.")

Prompt created successfully.


In [11]:
# ============================================================
# STEP 5.3: CREATE THE GEMINI MODEL
# ============================================================
# We use Gemini through LangChain.
#
# The GOOGLE_API_KEY was already loaded into the environment
# in Step 3, so we do not put the API key in this cell.
#
# Gemini 2.5 Flash is sufficient for this demonstration
# because we are focusing on prompt management and
# observability rather than model capability.
# ============================================================

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("Gemini model created successfully.")

Gemini model created successfully.


In [12]:
# ============================================================
# STEP 5.4: CREATE THE LANGCHAIN CHAIN
# ============================================================
# The pipe operator (|) connects the prompt to the model.
#
# Data flows as:
#
# Input
#   ↓
# Prompt
#   ↓
# Gemini
#   ↓
# Output
#
# Because LangSmith tracing has already been enabled,
# this execution can be recorded automatically by LangSmith.
# ============================================================

chain = prompt | model

print("LangChain chain created successfully.")

LangChain chain created successfully.


In [13]:
# ============================================================
# STEP 5.5: RUN THE FIRST LLM REQUEST
# ============================================================
# This is the first actual execution of our application.
#
# We send one customer question through:
#
#     Question → Prompt → Gemini → Response
#
# LangSmith should automatically record this execution
# because LANGSMITH_TRACING=true was configured earlier.
# ============================================================

question = "Can I return a used product?"

response = chain.invoke({
    "question": question
})

print("\nCustomer Question:")
print(question)

print("\nAssistant Response:")
print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Customer Question:
Can I return a used product?

Assistant Response:
[{'type': 'text', 'text': 'It depends on the item and its condition. Generally, used items can only be returned if they are defective, damaged, or covered under a specific satisfaction guarantee. \n\nTo check if your item qualifies for a return, please share your order number or purchase details, and I would be happy to look into it for you!', 'extras': {'signature': 'EuYVCuMVARFNMg9dCkjrQ31kdZ8GMtoF6AGPoOUmkRCuPzzNwY2Fb9H/802C14u0xq78NbFSu4XYT2To7NWsI6QyURg3Ko8eluNP0DV9kw2eWJMdmzM9j+tHtvxYu58CP7DctBqo6RLz5HAGmbnusyphNONoPk2jlZvGwkFaiCIEpc+HO75K37rMgOrWKS1HbIaECDFnyLahXmRDIXmaaZb2O+xXZ0M7wsSFfJQrFyqs/klQvCKshxYsoLdkUuv0TmArw1Qe2dVJsXMNH9QVeZ2+UHCMJksrdwhpYoABjpmUAqqh/fl2XzZ0GhpvuU57bMWmLwQYgrSyESqwJ1W1Zwbo2V7+qTXcT6E+iEtKrdTC33OdtlGAsDIc03KFCUqyWDeTPpjUpVEunBatgHDNHnQKJPAoyoGM4saqkrCkq8M5T4HHSwaVeHezoQg2+gAOE41+/OD5tR/I64Zr254ZHd5jfbDdJ6aZ0L9rmxI+0ZfCeYdIZR8Zb7xwkCl13zkwCEOFo4N5nlwZbKyz+Mt+dPNe99zw69FNraevcQD/joRNglO

In [14]:
# ============================================================
# STEP 6.1: CREATE A SMALL CUSTOMER-SERVICE POLICY
# ============================================================
# We create a fixed policy that the LLM must use when
# answering customer questions.
#
# Why are we doing this?
# ----------------------
# If we simply ask the model about returns, the model may
# answer using its general knowledge.
#
# That makes it difficult to determine whether an answer
# is actually correct.
#
# By providing a fixed policy, we can later measure:
#
#   1. Accuracy
#   2. Consistency
#   3. Hallucination
#   4. Instruction following
#   5. Differences between prompt versions
#
# This same policy will be reused throughout our
# LangSmith evaluation demonstration.
# ============================================================

policy = """
Customer Return Policy:

1. Unused items may be returned within 30 days of purchase
   when the customer provides the receipt.

2. Used items cannot normally be returned.

3. Used items may be returned within 30 days if the product
   is defective.

4. Returns without a receipt are not accepted.

5. Refunds are issued after the returned product is inspected.
"""

print("Customer return policy loaded successfully.")
print(policy)

Customer return policy loaded successfully.

Customer Return Policy:

1. Unused items may be returned within 30 days of purchase
   when the customer provides the receipt.

2. Used items cannot normally be returned.

3. Used items may be returned within 30 days if the product
   is defective.

4. Returns without a receipt are not accepted.

5. Refunds are issued after the returned product is inspected.



In [15]:
# ============================================================
# STEP 6.2: CREATE A POLICY-GROUNDED PROMPT
# ============================================================
# This prompt is more controlled than our original prompt.
#
# The model is explicitly instructed to:
#
#   - act as a customer-service assistant
#   - use the supplied policy
#   - avoid inventing information
#   - answer clearly
#
# {context} and {question} are variables.
# Their actual values will be supplied when the chain runs.
# ============================================================

policy_prompt = ChatPromptTemplate.from_template(
    """
    You are a customer service assistant.

    Answer the customer's question using ONLY the policy
    provided below.

    Do not invent or assume any policy that is not stated
    in the provided context.

    If the policy does not contain enough information to
    answer the question, clearly state that the information
    is not available in the policy.

    Customer Return Policy:
    {context}

    Customer Question:
    {question}

    Provide a clear and concise answer.
    """
)

print("Policy-grounded prompt created successfully.")

Policy-grounded prompt created successfully.


In [16]:
# ============================================================
# STEP 6.3: CREATE THE POLICY-GROUNDED CHAIN
# ============================================================
# The pipe operator connects:
#
#       policy_prompt → Gemini model
#
# The resulting chain accepts both:
#
#       context
#       question
#
# and sends the completed prompt to Gemini.
#
# Because LangSmith tracing is already enabled, this
# execution will also be visible in LangSmith.
# ============================================================

policy_chain = policy_prompt | model

print("Policy-grounded chain created successfully.")

Policy-grounded chain created successfully.


In [17]:
# ============================================================
# STEP 6.4: RUN THE POLICY-GROUNDED APPLICATION
# ============================================================
# We provide:
#
#   context  → our customer return policy
#   question → the customer's question
#
# The model should answer using the supplied policy.
# ============================================================

question = "Can I return a used product?"

response = policy_chain.invoke({
    "context": policy,
    "question": question
})

print("\nCustomer Question:")
print(question)

print("\nAssistant Response:")
print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Customer Question:
Can I return a used product?

Assistant Response:
[{'type': 'text', 'text': 'Normally, used items cannot be returned. However, a used product may be returned within 30 days of purchase if it is defective and you have the receipt.', 'extras': {'signature': 'Et4NCtsNARFNMg9BPxJBk/nOqAKaTHrqBbaogV3LDsDCsKgbEk4dzhRFsfLfbd2IoIBtvsRQiPQKESLDEep2pymrqIJ/9MCfdEwqsA8nIgNJsW+n/wxPH+9JOXPMXeMwzQBmLiapLhD55wKPSakiA73yNogCyuuTk8kALVWN4lFuGqtZadtXFpqjrVD0bLhS53yYGyh29dqSCqQG55CHOKwCl4GqczmgLVRvnXM72/sMgEupPvi5fG02Kpg/ikVrqT5ApxAGGiDx23nn03DDxoEYh09XgiVD3nlv0U0pkLo1CIjbaF9hJdwzydzg/e+0KmoXatiAGgWOlxyajoQQsmbgOC8Qa5bPbm5FBWDm918egtRJjzE1/aG8fnvtG8gqCxFRGd34qRc9vmV8LRLDo9KNQq5iNKHwtv2nruLXb/hL0F9AaUWiLKo3cLET+M9aQbLpEeJ8a1OlHbbP4GFwQSn8DNLpoPMBmVWfWXiQ5Pnk+xjX1ZL6cSrwt5vg+CCCPX++NILaqdvbMqJc/R6QKLpGkZ/cL3yO0t/dlfheJT4BEUMVM/DK0TzpPhbXmMQC1AVRBhwkV8l3b7Ipgucp0y4bdIFOxhy4KV8+Xkk2qQOMoTTPJIiv6EK/qGXwHzx+RapuFfUb/4JwCF0LMcLbn2v8qgRgvyURbhYZd+B0T5qNTpQG21whtgEJjjVOy6051DaeVXNW88fscGmNEDh

In [18]:
# ============================================================
# STEP 7.1: CREATE A DELIBERATELY WEAK PROMPT
# ============================================================
# This prompt is intentionally very simple:
#
#     "Answer the question."
#
# It does not tell the model:
#   - what role it should play
#   - what information it should use
#   - whether it should follow a policy
#   - whether it should avoid hallucinating
#   - how detailed the answer should be
#
# We are intentionally creating this weak version so that
# LangSmith can later help us inspect and compare its
# execution with an improved version.
# ============================================================

weak_prompt = ChatPromptTemplate.from_template(
    """
    Answer the question.

    Question:
    {question}
    """
)

print("Prompt V1 (weak prompt) created.")

Prompt V1 (weak prompt) created.


In [19]:
# ============================================================
# STEP 7.2: CONNECT PROMPT V1 TO THE GEMINI MODEL
# ============================================================
# The chain connects:
#
#       Prompt V1 → Gemini
#
# We are using the same Gemini model as before.
#
# This is important for a fair comparison:
# only the PROMPT will change.
# The model remains the same.
# ============================================================

weak_chain = weak_prompt | model

print("Prompt V1 chain created successfully.")

Prompt V1 chain created successfully.


In [20]:
# ============================================================
# STEP 7.3: RUN PROMPT V1
# ============================================================
# Notice that this weak prompt does NOT receive the policy.
#
# This is intentional.
#
# We want to observe what happens when the model is given
# only a question and very little instruction.
#
# LangSmith will automatically trace this execution because
# LangSmith tracing was enabled earlier.
# ============================================================

question = "Can I return a used product?"

response_v1 = weak_chain.invoke({
    "question": question
})

print("\nPrompt Version: V1")
print("Customer Question:")
print(question)

print("\nAssistant Response:")
print(response_v1.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 53.476958384s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '